Alumna: Claudia Valentín Alguacil
Titulación: Master en Big Data y Ciencia de datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV,GridSearchCV,StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

## Importación datos y transformación
En esta seccción se han importado los diferentes conjuntos de datos, han sido transformados para tener homogeneidad y unidos para el futuro analisis.

In [54]:
#Carga y transformación del dataset
df_demografia=pd.read_excel('01_demography_sppb_mmse.xlsx').rename(columns={'Subject ID':'Subject'})
df_antropometria=pd.read_excel('02_anthropometry.xlsx')
df_poma=pd.read_excel('03_poma_test.xlsx', header=1)
df_poma_media=df_poma.drop(columns=['Experience of the test administrator','Week']).groupby('Subject').mean().reset_index()
df_gait=pd.read_excel('05_gait_parameters.xlsx')
df_gait['Name']=df_gait['Name'].ffill()
df_gait_media=df_gait.rename(columns={'Name':'Subject'}).drop(columns=['Walk','Gait Duration after data crop']).groupby('Subject').mean().reset_index()
#Unificación del dataset
dataset=df_demografia.merge(df_antropometria,on='Subject',how='inner')
dataset=dataset.merge(df_poma_media,on='Subject',how='inner')
dataset=dataset.merge(df_gait_media,on='Subject',how='inner')

In [55]:
dataset.to_csv('dataset.csv', index=False)

In [56]:
dataset['Objetivo']=dataset['SPPB'].apply(lambda x: 1 if x < 10 else 0)

In [57]:
excluir = ['Subject','Sex','SPPB', 'Number of self-reported falls in the last month','Objetivo']
X = dataset.drop(columns=excluir)

Imputo valores nulos mediante la media de las variables

In [58]:
X = X.fillna(X.mean())
y = dataset['Objetivo']

In [59]:
semilla=42

In [64]:
X.columns

Index(['Age (Years)', 'Mass (Kg)', 'Heigth (m)', 'Body Mass Index (Kg/m^2)',
       'MMSE', 'Distance between iliac crests (mm)',
       'Distance between left iliac crest and left medial epicondyle (mm)',
       'Distance between right iliac crest and right medial epicondyle (mm)',
       'Left knee width (mm)', 'Right knee width (mm)',
       'Left ankle width (mm)', 'Right ankle width (mm)',
       '10. Initiation of gait ', '11.a Right foot passes left stance foot',
       '11.b Right foot completely clears floor ',
       '11.c Left foot passes right stance foot ',
       '11.d Left foot completely clears floor', '12. Step Symmetry ',
       '13. Step Continuity', '14. Path', '15. Trunk', '16. Walking Stance',
       'Total Gait Score', 'Left_Cadence ', 'Right_Cadence ',
       'Left_Walking_Speed', 'Right_Walking_Speed', 'Left_Stride_Time',
       'Right_Stride_Time ', 'Left_Step_Time', 'Right_Step_Time ',
       'Left_Opposite_Foot_Off', 'Right_Opposite_Foot_Off',
       'Left_O

# División
El conjunto del dataset se divide (80% entrenamiento , 20% prueba) para evitar sobreajuste y mantieniendo la proporcion de clases.

In [60]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=semilla, stratify=y)

In [61]:
print(y_train.value_counts(normalize=True) * 100)
print(y_test.value_counts(normalize=True) * 100)

Objetivo
0    62.857143
1    37.142857
Name: proportion, dtype: float64
Objetivo
0    66.666667
1    33.333333
Name: proportion, dtype: float64


El porcentaje esta parcialmente desbalancedado, aproximadamente 60% 40%.No supondría un gran riesgo en la aplicación de modelos de machine learning, por lo que no se ha realizado ningún cambio

In [10]:
cv = StratifiedKFold(n_splits=4,shuffle=True, random_state=semilla)

# Función de resultados
Esta función se aplicara a los diferentes modelos con la finalidad de encontrar el mejor.

In [11]:
def resultados(y_real, y_pred, modelo):
    accuracy = accuracy_score(y_real, y_pred)
    precision = precision_score(y_real, y_pred, average='binary')
    recall = recall_score(y_real, y_pred, average='binary')
    f1 = f1_score(y_real, y_pred, average='binary')
    conf_matrix = confusion_matrix(y_real, y_pred)
    class_report = classification_report(y_real, y_pred)
    print(f"Exactitud : {accuracy:.4f}")
    print(f"Precisión: {precision:.4f}")
    print(f"Sensibilidad : {recall:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("matriz de confusión:")
    print(conf_matrix)
    print("Classification Report:")
    print(class_report)
    tn, fp, fn, tp = conf_matrix.ravel()
    return {'Modelo': modelo,'Exactitud': accuracy, 'Precisión': precision, 'Sensibilidad': recall, 'F1-Score': f1,'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,}

# Random forest 

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [72]:
param_dist = {'n_estimators': [50, 100, 150,200], 'max_depth': [2, 3, 5], 'min_samples_split': [2, 5, 10],
              'min_samples_leaf': [1, 2, 4],'max_features': ['sqrt', 'log2'],'criterion': ['gini', 'entropy']}
random_forest = RandomForestClassifier(random_state=semilla)
rf_random = RandomizedSearchCV(estimator=random_forest, param_distributions=param_dist,
                               n_iter=50, cv=cv,scoring='recall',random_state=semilla, n_jobs=-1,verbose=1)
rf_random.fit(X_train, y_train)

Fitting 4 folds for each of 50 candidates, totalling 200 fits


,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'criterion': ['gini', 'entropy'], 'max_depth': [2, 3, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], ...}"
,n_iter,50
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [76]:
param_grid={'n_estimators':[30,50,100], 'max_depth':[3,5,7], 'min_samples_split':[3,5,7],
            'min_samples_leaf': [2, 4, 6], 'max_features': ['log2'],'criterion': ['entropy']}
rf=GridSearchCV(estimator=RandomForestClassifier(random_state=semilla), param_grid=param_grid,cv=cv,scoring='recall',n_jobs=-1,verbose=1)
rf.fit(X_train, y_train)

Fitting 4 folds for each of 81 candidates, totalling 324 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'criterion': ['entropy'], 'max_depth': [3, 5, ...], 'max_features': ['log2'], 'min_samples_leaf': [2, 4, ...], ...}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [77]:
modelo_rf = rf.best_estimator_
y_pred_test = modelo_rf.predict(X_test)
res_rf=resultados(y_test, y_pred_test,'Random Forest')

Exactitud : 0.7778
Precisión: 0.6667
Sensibilidad : 0.6667
F1-score: 0.6667
matriz de confusión:
[[5 1]
 [1 2]]
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.83      0.83         6
           1       0.67      0.67      0.67         3

    accuracy                           0.78         9
   macro avg       0.75      0.75      0.75         9
weighted avg       0.78      0.78      0.78         9



In [78]:
importancia = modelo_rf.feature_importances_
variable_significativa = pd.DataFrame({'Variable': X.columns, 'Importancia': importancia})
variable_significativa_rf = variable_significativa.sort_values(by='Importancia', ascending=False).head(5)
print("\nVariables más importantes para predecir el riesgo:")
print(variable_significativa_rf.to_string(index=False))


Variables más importantes para predecir el riesgo:
                Variable  Importancia
      Left_Walking_Speed     0.070833
          Right_Cadence      0.064832
     Right_Walking_Speed     0.058201
Body Mass Index (Kg/m^2)     0.050701
    Right_Double_Support     0.046225


# ExtraTreesClassifier

In [17]:
from sklearn.ensemble import ExtraTreesClassifier

In [65]:
param_dist = {'n_estimators': [50, 100, 150,200], 'max_depth': [2, 3, 5], 'min_samples_split': [2, 5, 10],
              'min_samples_leaf': [1, 2, 4],'max_features': ['sqrt', 'log2'],'criterion': ['gini', 'entropy']}
etc = RandomizedSearchCV(estimator=ExtraTreesClassifier(random_state=semilla), param_distributions=param_dist,
                               n_iter=50, cv=cv,scoring='recall',random_state=semilla, n_jobs=-1,verbose=1)
etc.fit(X_train, y_train)

Fitting 4 folds for each of 50 candidates, totalling 200 fits


,estimator,ExtraTreesCla...ndom_state=42)
,param_distributions,"{'criterion': ['gini', 'entropy'], 'max_depth': [2, 3, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], ...}"
,n_iter,50
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [69]:
param_grid = {'n_estimators': [50,100 ],'max_depth': [3, 5, 8],'min_samples_split': [2,3],
              'min_samples_leaf': [1, 2, 4], 'max_features': ['log2'], 'criterion': ['gini']}
treeClassifier = GridSearchCV(estimator=ExtraTreesClassifier(random_state=semilla),
                              param_grid=param_grid, cv=cv, scoring='recall', n_jobs=-1,verbose=1)
treeClassifier.fit(X_train, y_train)

Fitting 4 folds for each of 36 candidates, totalling 144 fits


,estimator,ExtraTreesCla...ndom_state=42)
,param_grid,"{'criterion': ['gini'], 'max_depth': [3, 5, ...], 'max_features': ['log2'], 'min_samples_leaf': [1, 2, ...], ...}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,50


In [70]:
modelo_tree = treeClassifier.best_estimator_
y_pred_test = modelo_tree.predict(X_test)
res_et=resultados(y_test, y_pred_test,'Extra Trees')

Exactitud : 0.7778
Precisión: 0.6667
Sensibilidad : 0.6667
F1-score: 0.6667
matriz de confusión:
[[5 1]
 [1 2]]
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.83      0.83         6
           1       0.67      0.67      0.67         3

    accuracy                           0.78         9
   macro avg       0.75      0.75      0.75         9
weighted avg       0.78      0.78      0.78         9



In [71]:
importancia = modelo_tree.feature_importances_
variable_significativa = pd.DataFrame({'Variable': X.columns, 'Importancia': importancia})
variable_significativa_et = variable_significativa.sort_values(by='Importancia', ascending=False).head(5)
print("\nVariables más importantes para predecir el riesgo de caída:")
print(variable_significativa_et.to_string(index=False))


Variables más importantes para predecir el riesgo de caída:
             Variable  Importancia
  Right_Walking_Speed     0.091649
     Right_Step_Time      0.055239
Right knee width (mm)     0.045023
        Left_Cadence      0.038705
     Left_Step_Length     0.038007


# Regresion Logistica
Es necesario estandarizar

In [114]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [115]:
from sklearn.linear_model import LogisticRegression

In [116]:
l_regresion=LogisticRegression(solver='liblinear', max_iter=5000, random_state=semilla)

In [131]:
param_dist = {'penalty': ['l1', 'l2'],'C': [0.1,0.3,0.5],'class_weight': [None, 'balanced']}

In [132]:
regresionds = RandomizedSearchCV(estimator=l_regresion,param_distributions=param_dist,
                         cv=cv,scoring='recall', n_jobs=-1,verbose=1)
regresionds.fit(X_train_scaled, y_train)

Fitting 4 folds for each of 10 candidates, totalling 40 fits


,estimator,LogisticRegre...r='liblinear')
,param_distributions,"{'C': [0.1, 0.3, ...], 'class_weight': [None, 'balanced'], 'penalty': ['l1', 'l2']}"
,n_iter,10
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [133]:
param_grid = {'penalty': ['l2'],'C': [0.1,0.3],'class_weight': [None]}

In [134]:
regresion =  GridSearchCV(estimator=l_regresion,param_grid=param_grid,
                         cv=cv,scoring='recall', n_jobs=-1,verbose=1)
regresion.fit(X_train_scaled, y_train)

Fitting 4 folds for each of 2 candidates, totalling 8 fits


,estimator,LogisticRegre...r='liblinear')
,param_grid,"{'C': [0.1, 0.3], 'class_weight': [None], 'penalty': ['l2']}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [135]:
modelo_RL = regresion.best_estimator_
y_pred_test = modelo_RL.predict(X_test_scaled )
res_lr=resultados(y_test, y_pred_test,'Regresión Logística')

Exactitud : 0.8889
Precisión: 0.7500
Sensibilidad : 1.0000
F1-score: 0.8571
matriz de confusión:
[[5 1]
 [0 3]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.83      0.91         6
           1       0.75      1.00      0.86         3

    accuracy                           0.89         9
   macro avg       0.88      0.92      0.88         9
weighted avg       0.92      0.89      0.89         9



In [136]:
perm = permutation_importance(modelo_RL, X_test_scaled, y_test, n_repeats=30,random_state=semilla, scoring='recall')
variable_significativa = pd.DataFrame({'Variable': X.columns, 'Importancia': perm.importances_mean})
variable_significativa_rl = variable_significativa.sort_values(by='Importancia', ascending=False).head(5)
print("\nVariables más importantes para predecir el riesgo:")
print(variable_significativa_rl.to_string(index=False))


Variables más importantes para predecir el riesgo:
                                                           Variable  Importancia
  Distance between left iliac crest and left medial epicondyle (mm)     0.188889
Distance between right iliac crest and right medial epicondyle (mm)     0.188889
                                                    Left_Limp_Index     0.055556
                                           Body Mass Index (Kg/m^2)     0.033333
                                                        Age (Years)     0.000000


# KNN

Se utilizan los datos normalizados.

In [97]:
from sklearn.neighbors import KNeighborsClassifier

In [137]:
param_dist = {'n_neighbors': [ 3,5,7],'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan', 'minkowski']}
knnds = RandomizedSearchCV(estimator=KNeighborsClassifier(),param_distributions=param_dist, cv=cv, scoring='recall',  n_jobs=-1,verbose=1)
knnds.fit(X_train_scaled, y_train)

Fitting 4 folds for each of 10 candidates, totalling 40 fits


,estimator,KNeighborsClassifier()
,param_distributions,"{'metric': ['euclidean', 'manhattan', ...], 'n_neighbors': [3, 5, ...], 'weights': ['uniform', 'distance']}"
,n_iter,10
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [138]:
param_grid = {'n_neighbors': [ 3,4,5],'weights': ['uniform'], 'metric': ['euclidean', 'minkowski']}
knn = GridSearchCV(estimator=KNeighborsClassifier(),param_grid=param_grid, cv=cv, scoring='recall',  n_jobs=-1,verbose=1)
knn.fit(X_train_scaled, y_train)

Fitting 4 folds for each of 6 candidates, totalling 24 fits


,estimator,KNeighborsClassifier()
,param_grid,"{'metric': ['euclidean', 'minkowski'], 'n_neighbors': [3, 4, ...], 'weights': ['uniform']}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,3


In [139]:
modelo_knn = knn.best_estimator_
y_pred_test = knn.predict(X_test_scaled)
res_knn=resultados(y_test, y_pred_test,'KNN')

Exactitud : 0.6667
Precisión: 0.5000
Sensibilidad : 0.6667
F1-score: 0.5714
matriz de confusión:
[[4 2]
 [1 2]]
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.67      0.73         6
           1       0.50      0.67      0.57         3

    accuracy                           0.67         9
   macro avg       0.65      0.67      0.65         9
weighted avg       0.70      0.67      0.68         9



In [140]:
perm = permutation_importance(modelo_knn, X_test_scaled, y_test, n_repeats=30,
                               random_state=semilla, scoring='recall')
variable_significativa = pd.DataFrame({'Variable': X.columns, 'Importancia': perm.importances_mean})
variable_significativa_knn = variable_significativa.sort_values(by='Importancia', ascending=False).head(5)
print("\nVariables más importantes para predecir el riesgo (importancia por permutación):")
print(variable_significativa_knn.to_string(index=False))


Variables más importantes para predecir el riesgo (importancia por permutación):
          Variable  Importancia
       Age (Years)          0.0
Left_Walking_Speed          0.0
  Left_Stride_Time          0.0
Right_Stride_Time           0.0
    Left_Step_Time          0.0


# SVC

In [102]:
from sklearn.svm import SVC

In [158]:
param_dist = { 'C':[0.1,0.5,1,2,5],'gamma': ['scale', 'auto', 0.1, 0.01],'kernel': ['linear', 'rbf']}
dist_svm = RandomizedSearchCV( estimator=SVC(), param_distributions=param_dist, scoring='recall', cv=cv,n_jobs=-1, verbose=1)
dist_svm.fit(X_train_scaled, y_train)

Fitting 4 folds for each of 10 candidates, totalling 40 fits


,estimator,SVC()
,param_distributions,"{'C': [0.1, 0.5, ...], 'gamma': ['scale', 'auto', ...], 'kernel': ['linear', 'rbf']}"
,n_iter,10
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [159]:
param_grid = { 'C':[3,5,7],'gamma': ['scale', 'auto',0.01],'kernel': ['linear', 'rbf']}
grid_svm = GridSearchCV( estimator=SVC(), param_grid=param_grid, scoring='recall', cv=cv,n_jobs=-1, verbose=1)
grid_svm.fit(X_train_scaled, y_train)

Fitting 4 folds for each of 18 candidates, totalling 72 fits


,estimator,SVC()
,param_grid,"{'C': [3, 5, ...], 'gamma': ['scale', 'auto', ...], 'kernel': ['linear', 'rbf']}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,5


In [160]:
modelo_svc = grid_svm.best_estimator_
y_pred_test = grid_svm.predict(X_test_scaled)
res_svc=resultados(y_test, y_pred_test,'SVC')

Exactitud : 0.5556
Precisión: 0.4000
Sensibilidad : 0.6667
F1-score: 0.5000
matriz de confusión:
[[3 3]
 [1 2]]
Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.50      0.60         6
           1       0.40      0.67      0.50         3

    accuracy                           0.56         9
   macro avg       0.57      0.58      0.55         9
weighted avg       0.63      0.56      0.57         9



In [161]:
perm = permutation_importance(modelo_svc, X_test_scaled, y_test, n_repeats=30,
                               random_state=semilla, scoring='recall')
variable_significativa = pd.DataFrame({'Variable': X.columns, 'Importancia': perm.importances_mean})
variable_significativa_svc = variable_significativa.sort_values(by='Importancia', ascending=False).head(5)
print("\nVariables más importantes para predecir el riesgo (importancia por permutación):")
print(variable_significativa_svc.to_string(index=False))



Variables más importantes para predecir el riesgo (importancia por permutación):
           Variable  Importancia
        Age (Years)          0.0
   Total Gait Score          0.0
 Left_Walking_Speed          0.0
Right_Walking_Speed          0.0
   Right_Step_Time           0.0


# XGBClassifier 

In [145]:
from xgboost import XGBClassifier

In [146]:
param_dist= {'n_estimators': [30,50,80,100], 'max_depth': [2, 3, 4],
              'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.7,0.9,1.0],
              'colsample_bytree': [0.6,0.8],'gamma': [0, 0.1, 0.5, 1.0]}
grid_xgb = RandomizedSearchCV(estimator= XGBClassifier(random_state=semilla, eval_metric='logloss'), param_distributions=param_dist, cv=cv,
                         scoring='recall', n_jobs=-1, verbose=1)
grid_xgb.fit(X_train, y_train)

Fitting 4 folds for each of 10 candidates, totalling 40 fits


,estimator,"XGBClassifier...ree=None, ...)"
,param_distributions,"{'colsample_bytree': [0.6, 0.8], 'gamma': [0, 0.1, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [2, 3, ...], ...}"
,n_iter,10
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [153]:
param_grid = {'n_estimators': [30,50], 'max_depth': [2,4 ],
              'learning_rate': [0.05, 0.1], 'subsample': [0.7,0.9],
              'colsample_bytree': [0.6,0.7],'gamma': [0.5, 1.0]}
grid_xgb = GridSearchCV(estimator= XGBClassifier(random_state=semilla, eval_metric='logloss'), param_grid=param_grid, cv=cv,
                         scoring='recall', n_jobs=-1, verbose=1)
grid_xgb.fit(X_train, y_train)

Fitting 4 folds for each of 64 candidates, totalling 256 fits


,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'colsample_bytree': [0.6, 0.7], 'gamma': [0.5, 1.0], 'learning_rate': [0.05, 0.1], 'max_depth': [2, 4], ...}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [154]:
modelo_xgb = grid_xgb.best_estimator_
y_pred_test = modelo_xgb.predict(X_test)
res_xgb = resultados(y_test, y_pred_test,'XGB')

Exactitud : 0.8889
Precisión: 0.7500
Sensibilidad : 1.0000
F1-score: 0.8571
matriz de confusión:
[[5 1]
 [0 3]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.83      0.91         6
           1       0.75      1.00      0.86         3

    accuracy                           0.89         9
   macro avg       0.88      0.92      0.88         9
weighted avg       0.92      0.89      0.89         9



In [155]:
importancia = modelo_xgb.feature_importances_
variable_significativa = pd.DataFrame({'Variable': X.columns, 'Importancia': importancia})
variable_significativa_xgb = variable_significativa.sort_values(by='Importancia', ascending=False).head(5)
print("\nVariables más importantes para predecir el riesgo de caída:")
print(variable_significativa_xgb.to_string(index=False))


Variables más importantes para predecir el riesgo de caída:
               Variable  Importancia
    Left_Double_Support     0.142490
    Right_Stride_Length     0.076010
     Left_Walking_Speed     0.061031
       Left_Stride_Time     0.056165
Right_Opposite_Foot_Off     0.055205


# Comparativa

In [156]:
comparativa = pd.DataFrame([res_rf, res_et, res_lr, res_knn, res_svc, res_xgb])
comparativa = comparativa.sort_values(by='Sensibilidad', ascending=False).reset_index(drop=True)
comparativa

,Modelo,Exactitud,Precisión,Sensibilidad,F1-Score,TN,FP,FN,TP
0,Regresión Logística,0.888889,0.750000,1.000000,0.857143,5,1,0,3
1,XGB,0.888889,0.750000,1.000000,0.857143,5,1,0,3
2,Random Forest,0.777778,0.666667,0.666667,0.666667,5,1,1,2
3,Extra Trees,0.777778,0.666667,0.666667,0.666667,5,1,1,2
4,KNN,0.666667,0.500000,0.666667,0.571429,4,2,1,2
5,SVC,0.555556,0.400000,0.666667,0.500000,3,3,1,2


# Top 5 Variables significativas de cada modelo

In [162]:
rf = variable_significativa_rf[['Variable', 'Importancia']].reset_index(drop=True).rename(columns={'Variable':'Random Forest'})
et = variable_significativa_et[['Variable', 'Importancia']].reset_index(drop=True).rename(columns={'Variable': 'ExtraTrees'})
knn = variable_significativa_knn[['Variable', 'Importancia']].reset_index(drop=True).rename(columns={'Variable':'KNN'})
svc = variable_significativa_svc[['Variable', 'Importancia']].reset_index(drop=True).rename(columns={'Variable': 'SVC'})
xgb = variable_significativa_xgb[['Variable', 'Importancia']].reset_index(drop=True).rename(columns={'Variable': 'XGB'})
rl=variable_significativa_rl[['Variable', 'Importancia']].reset_index(drop=True).rename(columns={'Variable': 'Regresión Logística'})
tabla_detallada = pd.concat([rf, et, knn, svc, xgb,rl], axis=1)
tabla_detallada.index = [f'Top {i+1}' for i in range(len(tabla_detallada))]

tabla_detallada.round(4)

,Random Forest,Importancia,ExtraTrees,Importancia,KNN,Importancia,SVC,Importancia,XGB,Importancia,Regresión Logística,Importancia
Top 1,Left_Walking_Speed,0.0708,Right_Walking_Speed,0.0916,Age (Years),0.0,Age (Years),0.0,Left_Double_Support,0.1425,Distance between left iliac crest and left med...,0.1889
Top 2,Right_Cadence,0.0648,Right_Step_Time,0.0552,Left_Walking_Speed,0.0,Total Gait Score,0.0,Right_Stride_Length,0.0760,Distance between right iliac crest and right m...,0.1889
Top 3,Right_Walking_Speed,0.0582,Right knee width (mm),0.0450,Left_Stride_Time,0.0,Left_Walking_Speed,0.0,Left_Walking_Speed,0.0610,Left_Limp_Index,0.0556
Top 4,Body Mass Index (Kg/m^2),0.0507,Left_Cadence,0.0387,Right_Stride_Time,0.0,Right_Walking_Speed,0.0,Left_Stride_Time,0.0562,Body Mass Index (Kg/m^2),0.0333
Top 5,Right_Double_Support,0.0462,Left_Step_Length,0.0380,Left_Step_Time,0.0,Right_Step_Time,0.0,Right_Opposite_Foot_Off,0.0552,Age (Years),0.0000
